# CPSC 483 — Housing Utilities

**Student Name:** *[Your Name]*  
**CWID:** *[Your CWID]*  
**Email:** *[your.email@csu.fullerton.edu]*  
**Course:** CPSC 483-02, Introduction to Machine Learning, Summer 2026  
**Instructor:** Dr. Anand Panangadan  
**Assignment:** Day 2 Class Exercise — DataFrame Subsetting (slide 33)  
**Date:** 2026-05-27

---

## Purpose
Practice subsetting a pandas `DataFrame` (the California housing dataset)
using both label-based (`.loc`) and position-based (`.iloc`) indexing.

Five subsetting tasks:
1. First 5 columns of `housing_full`
2. Only the `total_rooms` and `total_bedrooms` columns
3. Only rows where `population > 1000`
4. Only those columns **and** those rows (combine #2 and #3)
5. Only alternate rows (using `np.arange()` — see Task 5)


## 1. Imports

In [ ]:
# Standard data-science imports for this course.
import numpy as np
import pandas as pd


## 2. Load the data

Reuses the textbook's `load_housing_data()` helper so this notebook is
self-contained: running it from a fresh Colab kernel will download the
dataset on first use, then read it from disk on subsequent runs.


In [ ]:
from pathlib import Path
import tarfile
import urllib.request

def load_housing_data():
    """Download (if needed) and return the California housing dataset as a DataFrame."""
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()

# Confirm it loaded.  Expect 20,640 rows x 10 columns.
print("Shape:", housing_full.shape)
housing_full.head()


---
## Task 1 — First 5 columns

"First 5 columns" is a **positional** request (we want columns 0 through 4 regardless of their names),
so the idiomatic tool is `.iloc[rows, cols]` with `:` for all rows and `:5` for the first five columns.

- `:` on the row axis → keep every row.
- `:5` on the column axis → columns at positions 0, 1, 2, 3, 4 (end is exclusive in `.iloc`).


In [ ]:
first_5_cols = housing_full.iloc[:, :5]

print("Shape:", first_5_cols.shape)        # should be (20640, 5)
print("Columns:", list(first_5_cols.columns))
first_5_cols.head()


---
## Task 2 — Only columns `total_rooms` and `total_bedrooms`

Selecting by **column name** → use the bracket-of-list pattern: `df[["col1", "col2"]]`.

The **double brackets** matter: `df["total_rooms"]` (single brackets) returns a 1-D `Series`;
`df[["total_rooms"]]` (double brackets, a list inside) returns a 2-D `DataFrame`. We want a
DataFrame, so we always use the list form.


In [ ]:
rooms_cols = housing_full[["total_rooms", "total_bedrooms"]]

print("Shape:", rooms_cols.shape)          # should be (20640, 2)
rooms_cols.head()


---
## Task 3 — Only rows where `population > 1000`

Row filtering uses a **Boolean mask**: a Series of True/False values the same length as the
DataFrame. Wherever the mask is True, that row is kept.

We can write the mask in two equivalent ways:
- `housing_full[housing_full["population"] > 1000]`
- `housing_full.loc[housing_full["population"] > 1000]`

Both work. The `.loc` form is preferred when you also want to pick columns at the same time
(see Task 4).


In [ ]:
pop_mask = housing_full["population"] > 1000   # Boolean Series (one True/False per row)
big_pop_rows = housing_full[pop_mask]

print("Rows kept:", big_pop_rows.shape[0], "of", housing_full.shape[0])
print("Min population in result:", big_pop_rows["population"].min())  # sanity check: > 1000
big_pop_rows.head()


---
## Task 4 — Those columns AND those rows

Combine Tasks 2 and 3 with `.loc`: `df.loc[row_selector, column_selector]`.

- Row selector: the Boolean mask `population > 1000`.
- Column selector: the list `["total_rooms", "total_bedrooms"]`.

This is the single cleanest expression for the combined filter.


In [ ]:
rooms_for_big_pop = housing_full.loc[
    housing_full["population"] > 1000,
    ["total_rooms", "total_bedrooms"]
]

print("Shape:", rooms_for_big_pop.shape)   # rows matches Task 3, columns matches Task 2
rooms_for_big_pop.head()


---
## Task 5 — Alternate rows

The hint mentions `arrange()`, which is almost certainly **`np.arange()`** (NumPy's range
function — the spellings differ by one letter, and R/dplyr's `arrange()` is for *sorting*,
which doesn't match "alternate rows").

`np.arange(start, stop, step)` returns an array of integer positions. With step `2` we get
every other row index. Then `.iloc` picks those positions:


In [ ]:
# Method A — using np.arange() as the hint suggests
even_positions = np.arange(0, len(housing_full), 2)   # 0, 2, 4, 6, ...
alternate_rows = housing_full.iloc[even_positions]

print("Shape:", alternate_rows.shape)                 # ~half of 20640 = 10320
print("First few positions selected:", even_positions[:5])
alternate_rows.head()


Same result with the more idiomatic pandas slice form. `::2` means "from start to end, step 2."


In [ ]:
# Method B — the cleaner pandas idiom (same result)
alternate_rows_slice = housing_full.iloc[::2]

print("Shape:", alternate_rows_slice.shape)
print("Methods agree:", alternate_rows.equals(alternate_rows_slice))


---
## Sanity-check summary

A one-glance verification that every task produced the expected shape.


In [ ]:
summary = pd.DataFrame({
    "Task": [
        "1. First 5 columns",
        "2. total_rooms + total_bedrooms",
        "3. population > 1000",
        "4. (2) AND (3)",
        "5. alternate rows",
    ],
    "Rows":    [first_5_cols.shape[0], rooms_cols.shape[0],
                big_pop_rows.shape[0], rooms_for_big_pop.shape[0],
                alternate_rows.shape[0]],
    "Columns": [first_5_cols.shape[1], rooms_cols.shape[1],
                big_pop_rows.shape[1], rooms_for_big_pop.shape[1],
                alternate_rows.shape[1]],
})
summary
